# Scientific validation：两个可亲手做的小实验

这不是 AgentRivet/MadAgents 的复现，不使用 LLM、API、Rivet 或 MadGraph。所有数据均为教学构造，只依赖 Python 标准库。

目的：把论文中的两个问题变成你能亲自检查的逻辑：

1. 程序输出类型正确、能够运行，甚至总积分正确，都不充分保证物理分布正确。
2. Reviewer 与 Coder 共享同一个错误摘要时，内部一致不等于忠实于原文。

来源：AgentRivet §3（p.5）、§5.2（pp.9–11）；MadAgents Appendix C（pp.39–40）。公式推导与下面的简化实验是额外教学内容。

先看问题、预测输出，再运行。答案放在最后。

In [ ]:
from math import isclose, isfinite
from dataclasses import dataclass

print("Only Python standard-library modules are used.")

## 实验 1：二维 bin 展平时丢失物理面积

人为定义两维边界 `x_edges = [0, 1, 3]`、`y_edges = [0, 2, 5]`。单位暂记为 x-unit、y-unit。四个 bins 按 x 为外层、y 为内层展平。

`integrated_xs[i]` 是第 i 个 bin 内的积分截面（pb）。对应的平均微分截面是

$$f_i = \frac{\Delta\sigma_i}{\Delta x_i\Delta y_i}.$$

整数索引 `0, 1, 2, 3` 只是显示顺序，不是物理坐标。

**先预测：**若把整数索引的宽度 1 当成 bin 面积，会通过 Python 的数值类型检查吗？

In [ ]:
x_edges = [0.0, 1.0, 3.0]
y_edges = [0.0, 2.0, 5.0]
areas = [
    (x_edges[i + 1] - x_edges[i]) * (y_edges[j + 1] - y_edges[j])
    for i in range(len(x_edges) - 1)
    for j in range(len(y_edges) - 1)
]
integrated_xs = [2.0, 3.0, 4.0, 6.0]  # pb; invented teaching data
correct_density = [s / a for s, a in zip(integrated_xs, areas)]
wrong_density = [s / 1.0 for s in integrated_xs]  # deliberately wrong

def structurally_valid(values: list[float], expected_size: int) -> bool:
    return len(values) == expected_size and all(isfinite(v) for v in values)

print("bin areas:", areas)
print("correct density:", correct_density)
print("wrong density:", wrong_density)
print("correct structure:", structurally_valid(correct_density, 4))
print("wrong structure:", structurally_valid(wrong_density, 4))

**再预测：**用物理 bin 面积对两组输出积分，是否都能恢复总截面？这个检查所依据的信息，来自输出本身还是外部定义？

In [ ]:
def integrate_density(values: list[float], bin_areas: list[float]) -> float:
    if len(values) != len(bin_areas):
        raise ValueError("Each density needs exactly one bin area.")
    return sum(f * a for f, a in zip(values, bin_areas))

sigma_total = sum(integrated_xs)
print("Expected total:", sigma_total, "pb")
print("Correct integral:", integrate_density(correct_density, areas), "pb")
print("Wrong integral:", integrate_density(wrong_density, areas), "pb")
assert isclose(integrate_density(correct_density, areas), sigma_total)
assert not isclose(integrate_density(wrong_density, areas), sigma_total)

### 一个检查就够了吗？

假设系统发现总截面错误，于是把错误分布整体缩放，直到总积分正确。

**先预测：**这会让每个 bin 也正确吗？这是不是“指标通过，但目标没实现”的一个反例？

In [ ]:
scale = sigma_total / integrate_density(wrong_density, areas)
renormalized_wrong = [scale * v for v in wrong_density]

print("Rescaled wrong density:", [round(v, 6) for v in renormalized_wrong])
print("Rescaled integral:", integrate_density(renormalized_wrong, areas), "pb")
print("All bins correct:", all(isclose(a, b) for a, b in zip(renormalized_wrong, correct_density)))
assert isclose(integrate_density(renormalized_wrong, areas), sigma_total)
assert not all(isclose(a, b) for a, b in zip(renormalized_wrong, correct_density))

## 实验 2：忠实执行了错误摘要

论文原文、摘要、代码、review 不是同一层。

下面人为设定原始要求是 `pT > 25 GeV`，摘要却变成 `pT > 20 GeV`。代码完整实现摘要，review 也只参考摘要。

这不是某个模型的真实输出，只是从 AgentRivet 的信息结构出发构造的可检查反例。

**先预测：**按摘要验代码会给什么结果？按原始要求验代码呢？

In [ ]:
@dataclass(frozen=True)
class SelectionSpec:
    pt_threshold: float
    unit: str = "GeV"

    def structurally_valid(self) -> bool:
        return isfinite(self.pt_threshold) and self.pt_threshold >= 0 and self.unit == "GeV"

original_spec = SelectionSpec(25.0)
extracted_summary = SelectionSpec(20.0)  # deliberately incorrect extraction
pt_values = [19.0, 22.0, 25.0, 26.0, 40.0]

def implementation(pt: float) -> bool:
    return pt > extracted_summary.pt_threshold

def agrees_with(spec: SelectionSpec, inputs: list[float]) -> bool:
    return all(implementation(pt) == (pt > spec.pt_threshold) for pt in inputs)

print("Summary structure valid:", extracted_summary.structurally_valid())
print("Reviewer using summary accepts:", agrees_with(extracted_summary, pt_values))
print("Independent original-spec check accepts:", agrees_with(original_spec, pt_values))
print("pT | code decision | original-spec decision")
for pt in pt_values:
    print(f"{pt:4.0f} | {str(implementation(pt)):13s} | {pt > original_spec.pt_threshold}")
assert agrees_with(extracted_summary, pt_values)
assert not agrees_with(original_spec, pt_values)

### 自己改一个地方

在下一格任选一种改动：

- 只测 `pT = [19, 40]`：为什么错误代码会被这个独立测试放过？
- 把阈值改回 25，但将 `>` 改成 `>=`：怎样选输入才能发现边界错误？
- 设想原论文没有写阈值：现在是否仍有依据把 20 判成错误？

不需要一次做完。重点是设计一个能区分正确与错误实现的输入。

In [ ]:
# Write a small test here.


## 参考思路（先尝试再看）

**实验 1：**结构检查对两组输出都通过；正确积分是 15 pb，错误积分是 65 pb。将错误分布缩放后可以恢复 15 pb，但 bin-by-bin 形状仍然不正确。因此结构、运行、总积分和局部物理行为是不同层次的检查。

**实验 2：**代码与错误摘要完全一致，会通过只依赖摘要的 reviewer。独立原始要求能揭示错误，但仅使用 [19, 40] 也不能区分两个阈值；测试数据必须覆盖有区分力的区域。检查 `>`/`>=` 需要阈值处输入。

**原文缺失信息时：**没有额外证据，就不能指定唯一正确阈值。应区分“原文含糊”与“明确定义被抽取错”。

**回到论文：**这些例子解释为什么 reviewer approval、execution success、run-to-run agreement 或某个总指标，都不是科学正确性的充分条件。它们不测量两个原系统的实际错误率，也没有证明提出的机制具备发表级新颖性。

## 一句话记录

我运行/修改了：

我能解释的错误链条是：

这让我想追问原文的一个问题是：
